# ML-09 — Validation Audit, Error Analysis, and Client Holdout Generalization

This notebook preserves our exact Week-6 validation audit on `w05_ml_practice_dataset.csv` ($N = 100$)—including the 4 false-positive test errors (`item_100`, `item_067`, `item_079`, `item_063`)—and extends the validation to the 30,000-row FlyRank starter dataset under a **Client-Grouped Holdout Split** (26 training clients / 6 unseen test clients).

## 1. Exact Week-6 Error Analysis on the W05 Practice Test Split ($n = 20$)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_repo_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if (p / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return p
    return cur

REPO_ROOT = find_repo_root()
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

df_w05 = pd.read_csv(REPO_ROOT / "work" / "data" / "w05_ml_practice_dataset.csv")
features = ["impressions", "clicks", "staleness_days", "position"]
X = df_w05[features]
y = df_w05["target"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

test_df = X_test.copy()
test_df["test_pos"] = range(len(test_df))
test_df["item_id"] = df_w05.loc[X_test.index, "item_id"].apply(lambda v: f"item_{int(v):03d}" if str(v).isdigit() else str(v))
test_df["actual"] = y_test.values
test_df["predicted"] = y_pred
test_df["pred_prob"] = np.round(y_prob, 4)

errors = test_df[test_df["actual"] != test_df["predicted"]]
print(f"Total Test Errors: {len(errors)} out of {len(test_df)} (4 False Positives, 0 False Negatives):")
print(errors[["test_pos", "item_id", "impressions", "clicks", "staleness_days", "position", "actual", "predicted", "pred_prob"]].to_string(index=False))


Total Test Errors: 4 out of 20 (4 False Positives, 0 False Negatives):
 test_pos  item_id  impressions  clicks  staleness_days  position  actual  predicted  pred_prob
        5 item_100          904     740              22         2       0          1     0.5336
        9 item_067         1095     460              24         2       0          1     0.5379
       10 item_079          479     476              25        10       0          1     0.5557
       12 item_063         4898     749              19         7       0          1     0.7891


## 2. Client-Grouped Holdout Validation on the 30,000-Row Starter Dataset

*We compare the Week-4 rule baseline, the raw 4-feature Logistic Regression, and the 7-feature log-scaled pre-decision Logistic Regression across 6 unseen test clients ($n_{\text{test}} = 2,325$).* 

In [2]:
import json

with open(REPO_ROOT / "work" / "outputs" / "capstone_metrics.json", "r", encoding="utf-8") as f:
    M = json.load(f)

grp = M["flyrank_30k_evaluation"]["splits"]["client_grouped_holdout"]
rows = []
for key, label in [
    ("w04_rule_baseline", "Week-4 Rule Baseline (score >= 3)"),
    ("starter_ref_baseline", "Starter Reference Baseline"),
    ("logreg_4feat_raw", "Logistic Regression (4 raw features)"),
    ("logreg_7feat_predecision", "Logistic Regression (7 pre-decision features, log-scaled)"),
]:
    m = grp[key]
    rows.append({
        "Method": label,
        "Accuracy": m["accuracy"],
        "F1": m["f1"],
        "Precision": m["precision"],
        "Recall": m["recall"],
        "ROC_AUC": m["roc_auc"],
        "Precision@20": m["precision_at_20"],
        "Precision@50": m["precision_at_50"],
    })

print(f"Client-Grouped Holdout Results (n_train={grp['train_rows']}, n_test={grp['test_rows']}, test_base_rate={grp['test_positive_rate']}):")
print(pd.DataFrame(rows).to_string(index=False))


Client-Grouped Holdout Results (n_train=27675, n_test=2325, test_base_rate=0.391):
                                                   Method  Accuracy     F1  Precision  Recall  ROC_AUC  Precision@20  Precision@50
                        Week-4 Rule Baseline (score >= 3)    0.4817 0.4356     0.3793  0.5116   0.4821          0.40          0.50
                               Starter Reference Baseline    0.6086 0.2743     0.4986  0.1892   0.6269          0.15          0.24
                     Logistic Regression (4 raw features)    0.3897 0.5256     0.3775  0.8647   0.4191          0.30          0.28
Logistic Regression (7 pre-decision features, log-scaled)    0.6688 0.5595     0.5828  0.5380   0.7144          0.70          0.62


## 3. Representative Client-Holdout Errors (False Positives & False Negatives)

In [3]:
err_holdout = M["flyrank_30k_evaluation"]["error_analysis_client_holdout"]
print("Top 3 False Positives on Unseen Clients:")
print(pd.DataFrame(err_holdout["false_positives_top3"])[["content_id", "client_id", "impressions", "clicks", "staleness_days", "position", "trend_direction", "pred_prob"]].to_string(index=False))

print("\nTop 3 False Negatives on Unseen Clients:")
print(pd.DataFrame(err_holdout["false_negatives_top3"])[["content_id", "client_id", "impressions", "clicks", "staleness_days", "position", "trend_direction", "pred_prob"]].to_string(index=False))


Top 3 False Positives on Unseen Clients:
          content_id         client_id  impressions  clicks  staleness_days  position trend_direction  pred_prob
content_3e79eaafc89d client_f74efabef1         8779       0              20      11.6          stable   0.915925
content_8fdbff16a886 client_f74efabef1         3863       0              20      12.0             new   0.905985
content_25ebfb5aa399 client_f74efabef1         4984       0              20      14.4          stable   0.886778

Top 3 False Negatives on Unseen Clients:
          content_id         client_id  impressions  clicks  staleness_days  position trend_direction  pred_prob
content_86748254b6bf client_d4735e3a26            1       0              20      90.0            down   0.025259
content_50daa4cd4375 client_d4735e3a26            4       0              20      65.8            down   0.099943
content_929f003731a9 client_d4735e3a26            1       0              20      15.0            down   0.102297


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`